# Sampling from BayesianFNN Model

This notebook demonstrates sampling from a **BayesianFNN** model loaded from ONNX format.

We visualize individual Monte Carlo samples to understand the model's uncertainty and decision boundaries.


In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split

from uqbench.data.loaders import generate_toy_dataset
from uqbench.models import BayesianFNN
from uqbench.utils.model_export import load_onnx_model
from uqbench.utils.toy_visualization import plot_single_sample_predictions

# Set style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

# Create figures directory for saving plots
figures_dir = Path("figures")
figures_dir.mkdir(exist_ok=True)
print(f"Figures will be saved to: {figures_dir.absolute()}")

# Path to saved model (new structure: model_name/model.onnx)
save_model_directory = Path("models")
onnx_model_path = save_model_directory / "bayesianfnn" / "model.onnx"

# Set random seed for reproducibility
seed = 42
rng = jax.random.PRNGKey(seed)
np.random.seed(seed)

print("Imports complete!")

## 1. Load Dataset

We use the same toy dataset to test the loaded model.


In [ ]:
# Generate dataset (same as training)
DATASET_OVERLAP = 4.0
X, y, y_onehot = generate_toy_dataset(
    n_samples=15000, overlap=DATASET_OVERLAP, seed=seed
)

# Split into train/test (same split as training)
X_train, X_test, y_train, y_test, y_train_onehot, y_test_onehot = train_test_split(
    X, y, y_onehot, test_size=0.9, random_state=seed, stratify=y
)

print(f"Test set: {X_test.shape[0]} samples")

## 2. Load Model from ONNX

Load the BayesianFNN model that was exported to ONNX format.


In [ ]:
# Load ONNX model
if not onnx_model_path.exists():
    raise FileNotFoundError(
        f"ONNX model not found at {onnx_model_path}. "
        "Please run train_analyze_posterior.ipynb first to export the model."
    )

onnx_session, metadata = load_onnx_model(onnx_model_path)
print(f"Loaded ONNX model: {metadata.get('model_name', 'BayesianFNN')}")
print(f"Input shape: {metadata.get('input_shape', 'unknown')}")
print(f"Output shape: {metadata.get('output_shape', 'unknown')}")

# Get input/output names from ONNX session
input_name = onnx_session.get_inputs()[0].name
output_name = onnx_session.get_outputs()[0].name
print(f"ONNX input name: {input_name}")
print(f"ONNX output name: {output_name}")

# Test ONNX inference
test_input = np.zeros((1, 2), dtype=np.float32)
onnx_output = onnx_session.run([output_name], {input_name: test_input})[0]
print(f"ONNX test output shape: {onnx_output.shape}")

# For MC sampling visualization, we still need the JAX model
# since ONNX models are deterministic. We'll recreate it for sampling.
print("\nNote: ONNX models are deterministic. For MC sampling visualization,")
print(
    "we'll use the JAX model structure. The ONNX model is used for single predictions."
)

# Recreate the model (same architecture as training) for MC sampling
bayesian_fnn_model = BayesianFNN(
    hidden_dims=(32, 32, 32), num_classes=2, beta=0.001, posterior_std_init=0.1
)

# Initialize model to get parameter structure
rng, init_rng = jax.random.split(rng)
dummy_input = jnp.zeros((1, 2), dtype=jnp.float32)
bayesian_fnn_params = bayesian_fnn_model.init(init_rng, dummy_input, rng, training=True)

## 3. Sample Individual Predictions

Visualize individual Monte Carlo samples from the BayesianFNN model to see the variation in decision boundaries.


In [ ]:
# Check individual predictions (single sample, not averaged) to see if model is non-linear
print("BayesianFNN - Individual Samples:")
plot_single_sample_predictions(
    bayesian_fnn_model,
    bayesian_fnn_params,
    X_test,
    y_test,
    "BayesianFNN",
    n_samples_to_show=5,
    seed=seed,
    figures_dir=figures_dir,
)